# Attacking a Pneumonia Detector — FGSM & PGD on real chest X-rays

**The medical problem:** pneumonia is usually diagnosed partly from chest X-rays, and AI models that read X-rays are one of the most widely studied medical AI tasks. This notebook trains a small detector on **PneumoniaMNIST** — real, de-identified chest X-rays of children (MedMNIST dataset), labeled **NORMAL** or **PNEUMONIA** — and then attacks it with FGSM and PGD.

**The two scary outcomes you will see with your own eyes:**

1. **Missed disease (false negative):** an X-ray the model correctly flags as PNEUMONIA, with a tiny invisible noise layer added, is now called **NORMAL**. In a clinic, that's a patient sent home untreated.
2. **False alarm (false positive):** a healthy X-ray, with noise added, is now called **PNEUMONIA**. Unnecessary alarm, tests, and stress.

**What's inside:** example X-rays → train detector → both flip demos → interactive epsilon slider → accuracy vs epsilon → targeted attack → FGSM vs PGD → shareable example gallery.

> Educational robustness demo on public de-identified data. Not a clinical tool.

---

## How to run this in Google Colab (step by step)

1. This file is at `Adversarial MNIST FGM/notebooks/PneumoniaMNIST_FGSM_Colab.ipynb` on your Mac.
2. Go to **https://colab.research.google.com**, sign in with any Google account (free).
3. **File → Upload notebook** → drag this file in.
4. *Optional speed boost:* **Runtime → Change runtime type → T4 GPU → Save**.
5. **Runtime → Run all** (or run cells one by one with **Shift+Enter** — recommended).
6. Wait ~5 min. Everything (dataset, training, attacks) happens inside this one notebook — nothing to install on your Mac.
7. Play with the epsilon slider in section 5, then **File → Save a copy in Drive** to keep it.

If the session disconnects, just `Runtime → Run all` again. Set `USE_DRIVE = True` in setup to auto-save figures/model to Google Drive.

## 0. Setup

Colab has PyTorch preinstalled; the only install is `medmnist` (the dataset package, few MB).

In [ ]:
# medmnist (the dataset package) is the only thing not preinstalled on Colab:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist"], check=True)

import json, random, csv, io
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

import ipywidgets as widgets
from IPython.display import display, clear_output
import medmnist  # dataset + model zoo package

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cpu":
    print("tip: Runtime -> Change runtime type -> T4 GPU is ~10x faster (CPU works too)")

USE_DRIVE = False  # set True to keep artifacts across Colab sessions
ART = Path("artifacts")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ART = Path("/content/drive/MyDrive/pneumonia-attack/artifacts")
for d in ["checkpoints", "figures", "metrics"]:
    (ART / d).mkdir(parents=True, exist_ok=True)
print("artifacts ->", ART.resolve())

## 1. The medical data — real chest X-rays

**PneumoniaMNIST** (part of MedMNIST): 5,856 de-identified chest X-ray images of children aged 1–5, each labeled by physicians as **NORMAL** or **PNEUMONIA**. Images are 28x28 grayscale, pixels in [0, 1] — the exact format our FGSM/PGD code expects.

- 4,708 train / 524 validation / 624 test images
- Imbalanced: there are more pneumonia cases than normal ones (realistic for a hospital dataset)

In [ ]:
CLASS_NAMES = {0: "NORMAL", 1: "PNEUMONIA"}

def to_tensor(img):
    # converts whatever medmnist gives us (numpy array or PIL image) to a (1, 28, 28) tensor in [0, 1]
    arr = np.array(img)
    if arr.ndim == 2:
        arr = arr[..., None]
    return torch.from_numpy(arr).float().permute(2, 0, 1) / 255.0

train_ds = medmnist.PneumoniaMNIST(split="train", download=True, transform=to_tensor)
val_ds = medmnist.PneumoniaMNIST(split="val", download=True, transform=to_tensor)
test_ds = medmnist.PneumoniaMNIST(split="test", download=True, transform=to_tensor)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)

tr_y = train_ds.labels.squeeze()
te_y = test_ds.labels.squeeze()
print(f"train: {len(train_ds)} X-rays  ({(tr_y == 0).sum()} normal / {(tr_y == 1).sum()} pneumonia)")
print(f"test:  {len(test_ds)} X-rays  ({(te_y == 0).sum()} normal / {(te_y == 1).sum()} pneumonia)")

normals = np.where(te_y == 0)[0]
pneumonias = np.where(te_y == 1)[0]

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j in range(4):
    img, _ = test_ds[normals[j]]
    axes[0, j].imshow(img[0], cmap="gray", vmin=0, vmax=1); axes[0, j].set_title("NORMAL", fontsize=10); axes[0, j].axis("off")
    img, _ = test_ds[pneumonias[j]]
    axes[1, j].imshow(img[0], cmap="gray", vmin=0, vmax=1); axes[1, j].set_title("PNEUMONIA", fontsize=10); axes[1, j].axis("off")
fig.suptitle("Real chest X-rays from the test set (never seen during training)")
plt.tight_layout()
fig.savefig(ART / "figures" / "xray_examples.png", dpi=150)
plt.show()

## 2. Train the pneumonia detector (~1–2 min)

A small CNN (the same one as the digits project, just 2 output classes). Target: ~85–92% test accuracy — real X-rays are harder than digits.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28 -> 14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14 -> 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(0.25), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

@torch.no_grad()
def evaluate(loader, net):
    net.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE).view(-1)
        correct += (net(images).argmax(1) == labels).sum().item()
        total += labels.numel()
    return correct / total

@torch.no_grad()
def confusion_counts(loader, net):
    tp = fp = tn = fn = 0
    net.eval()
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE).view(-1)
        preds = net(images).argmax(1)
        tp += ((preds == 1) & (labels == 1)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()
    return tp, fp, tn, fn

model = SmallCNN(num_classes=2).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 8
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE).view(-1)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running += loss.item() * labels.size(0)
    test_acc = evaluate(test_loader, model)
    history.append((running / len(train_ds), test_acc))
    print(f"epoch {epoch}: train loss {history[-1][0]:.4f} | test acc {test_acc * 100:.2f}%")

CLEAN_ACC = history[-1][1]
tp, fp, tn, fn = confusion_counts(test_loader, model)
print(f"\nclean test accuracy: {CLEAN_ACC * 100:.2f}%")
print(f"of the {tp + fn} real pneumonia cases: {tp} caught, {fn} MISSED")
print(f"of the {tn + fp} healthy patients: {tn} correctly cleared, {fp} false alarms")

plt.figure(figsize=(6, 3))
plt.plot([a * 100 for _, a in history], marker="o")
plt.xlabel("epoch"); plt.ylabel("test accuracy (%)")
plt.title("Detector training"); plt.grid(alpha=0.3)
plt.show()

torch.save({"model": model.state_dict(), "test_acc": CLEAN_ACC}, ART / "checkpoints" / "pneumonia_cnn.pt")
with open(ART / "metrics" / "train_history.json", "w") as f:
    json.dump([{"epoch": i + 1, "train_loss": l, "test_acc": a} for i, (l, a) in enumerate(history)], f, indent=2)
print("saved ->", ART / "checkpoints" / "pneumonia_cnn.pt")

## 3. The FGSM attack — same 5 lines of math as the digits project

```
x_adv = x + eps * sign(grad_x loss)   # untargeted: just flip the diagnosis
```

Freeze the weights, compute the gradient of the loss **with respect to the pixels**, move every pixel by exactly `eps` in the gradient's direction, clip back to [0, 1]. To your eye: same X-ray. To the model: a different patient.

In [ ]:
for p in model.parameters():
    p.requires_grad_(False)
model.eval()

def fgsm_attack(model, images, labels, epsilon, targeted=False, loss_fn=None):
    # labels = true class (untargeted) or the desired class (targeted)
    loss_fn = loss_fn or nn.CrossEntropyLoss()
    images = images.clone().detach().requires_grad_(True)
    loss = loss_fn(model(images), labels)
    (grad,) = torch.autograd.grad(loss, images)
    step = epsilon * grad.sign()
    perturbed = images - step if targeted else images + step
    return perturbed.clamp(0.0, 1.0).detach()

def predict_probs(x):
    with torch.no_grad():
        return model(x).softmax(1)[0]

## 4. Examples — the two scary flips

**Flip 1: pneumonia → NORMAL (false negative).** The model sees a sick patient and, thanks to the noise, clears them. A real system that auto-triages X-rays could send this patient home.

In [ ]:
EPS_DEMO = 0.3

def find_flip(indices, from_class, to_class, eps, cap=200):
    for i in indices[:cap]:
        img, _ = test_ds[int(i)]
        x = img.unsqueeze(0).to(DEVICE)
        if predict_probs(x).argmax().item() != from_class:
            continue
        adv = fgsm_attack(model, x, torch.tensor([from_class], device=DEVICE), eps)
        if predict_probs(adv).argmax().item() == to_class:
            return x, adv
    return None

flip1 = find_flip(pneumonias, 1, 0, EPS_DEMO)
assert flip1 is not None, "no pneumonia->normal flip found in first 200 pneumonia X-rays"
x1, adv1 = flip1
p_clean, p_adv = predict_probs(x1), predict_probs(adv1)
print(f"true diagnosis: PNEUMONIA")
print(f"  clean:        model says {CLASS_NAMES[p_clean.argmax().item()]} ({p_clean.max() * 100:.1f}%)")
print(f"  adversarial:  model says {CLASS_NAMES[p_adv.argmax().item()]} ({p_adv.max() * 100:.1f}%)   <- missed disease, eps={EPS_DEMO}")

fig = plt.figure(figsize=(10, 3.4))
ax1 = fig.add_subplot(1, 3, 1); ax1.imshow(x1[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
ax1.set_title(f"ORIGINAL X-ray — true PNEUMONIA\nmodel: {CLASS_NAMES[p_clean.argmax().item()]} ({p_clean.max() * 100:.1f}%)", fontsize=9)
ax2 = fig.add_subplot(1, 3, 2); ax2.imshow(((adv1 - x1)[0, 0] / EPS_DEMO).cpu(), cmap="gray", vmin=-1, vmax=1)
ax2.set_title(f"NOISE (amplified {1 / EPS_DEMO:.0f}x)\ninvisible to you", fontsize=9)
ax3 = fig.add_subplot(1, 3, 3); ax3.imshow(adv1[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
ax3.set_title(f"ADVERSARIAL\nmodel: {CLASS_NAMES[p_adv.argmax().item()]} ({p_adv.max() * 100:.1f}%)", fontsize=9)
for ax in fig.axes:
    ax.axis("off")
fig.suptitle("FLIP 1 — missed pneumonia (false negative)")
plt.tight_layout()
fig.savefig(ART / "figures" / "flip1_pneumonia_to_normal.png", dpi=150)
plt.show()

**Flip 2: NORMAL → pneumonia (false alarm).** The reverse: a healthy child flagged as sick.

In [ ]:
flip2 = find_flip(normals, 0, 1, EPS_DEMO)
assert flip2 is not None, "no normal->pneumonia flip found in first 200 normal X-rays"
x2, adv2 = flip2
p_clean2, p_adv2 = predict_probs(x2), predict_probs(adv2)
print(f"true diagnosis: NORMAL")
print(f"  clean:        model says {CLASS_NAMES[p_clean2.argmax().item()]} ({p_clean2.max() * 100:.1f}%)")
print(f"  adversarial:  model says {CLASS_NAMES[p_adv2.argmax().item()]} ({p_adv2.max() * 100:.1f}%)   <- false alarm, eps={EPS_DEMO}")

fig = plt.figure(figsize=(10, 3.4))
ax1 = fig.add_subplot(1, 3, 1); ax1.imshow(x2[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
ax1.set_title(f"ORIGINAL X-ray — true NORMAL\nmodel: {CLASS_NAMES[p_clean2.argmax().item()]} ({p_clean2.max() * 100:.1f}%)", fontsize=9)
ax2 = fig.add_subplot(1, 3, 2); ax2.imshow(((adv2 - x2)[0, 0] / EPS_DEMO).cpu(), cmap="gray", vmin=-1, vmax=1)
ax2.set_title(f"NOISE (amplified {1 / EPS_DEMO:.0f}x)", fontsize=9)
ax3 = fig.add_subplot(1, 3, 3); ax3.imshow(adv2[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
ax3.set_title(f"ADVERSARIAL\nmodel: {CLASS_NAMES[p_adv2.argmax().item()]} ({p_adv2.max() * 100:.1f}%)", fontsize=9)
for ax in fig.axes:
    ax.axis("off")
fig.suptitle("FLIP 2 — false pneumonia alarm (false positive)")
plt.tight_layout()
fig.savefig(ART / "figures" / "flip2_normal_to_pneumonia.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 2.6))
w = 0.35
ax.bar(np.arange(2) - w / 2, p_clean2.cpu(), w, label="clean X-ray")
ax.bar(np.arange(2) + w / 2, p_adv2.cpu(), w, label="adversarial X-ray")
ax.set_xticks(range(2)); ax.set_xticklabels(["NORMAL", "PNEUMONIA"])
ax.set_ylabel("probability"); ax.set_title("the model's diagnosis changes")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.show()

## 5. Interactive playground

- **image #** — any of the 624 test X-rays (0 to 623)
- **epsilon** — attack strength: 0.0 = clean, 0.3 = most X-rays flip, 0.4 = strong

Watch the diagnosis text and the probability bars flip as you drag.

In [ ]:
img_picker = widgets.IntSlider(value=int(pneumonias[0]), min=0, max=len(test_ds) - 1, step=1,
                               description="image #", style={"description_width": "initial"})
eps_picker = widgets.FloatSlider(value=0.2, min=0.0, max=0.5, step=0.05,
                                 description="epsilon", readout_format=".2f",
                                 style={"description_width": "initial"})
out = widgets.Output()

def render(idx, eps):
    img, label = test_ds[idx]
    label = int(label)
    x = img.unsqueeze(0).to(DEVICE)
    clean_p = predict_probs(x)
    adv = fgsm_attack(model, x, torch.tensor([label], device=DEVICE), eps) if eps > 0 else x
    adv_p = predict_probs(adv)
    flipped = adv_p.argmax().item() != label

    with out:
        clear_output(wait=True)
        fig = plt.figure(figsize=(10, 3.2))
        ax1 = fig.add_subplot(1, 3, 1); ax1.imshow(img[0], cmap="gray", vmin=0, vmax=1)
        ax1.set_title(f"original (true {CLASS_NAMES[label]})\nmodel: {CLASS_NAMES[clean_p.argmax().item()]} ({clean_p.max() * 100:.1f}%)", fontsize=9)
        ax2 = fig.add_subplot(1, 3, 2)
        if eps > 0:
            ax2.imshow(((adv - x)[0, 0] / eps).cpu(), cmap="gray", vmin=-1, vmax=1)
        ax2.set_title(f"noise (eps={eps:.2f})", fontsize=9)
        ax3 = fig.add_subplot(1, 3, 3); ax3.imshow(adv[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
        ax3.set_title(f"adversarial\nmodel: {CLASS_NAMES[adv_p.argmax().item()]} ({adv_p.max() * 100:.1f}%)", fontsize=9)
        for ax in fig.axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()

        if flipped:
            print(f"DIAGNOSIS CHANGED — true {CLASS_NAMES[label]}, model now says "
                  f"{CLASS_NAMES[adv_p.argmax().item()]} ({adv_p.max() * 100:.1f}% sure)")
        else:
            print(f"true {CLASS_NAMES[label]}, model still says {CLASS_NAMES[label]} — push epsilon higher")

        fig2, ax = plt.subplots(figsize=(6, 2.2))
        w = 0.35
        ax.bar(np.arange(2) - w / 2, clean_p.cpu(), w, label="clean")
        ax.bar(np.arange(2) + w / 2, adv_p.cpu(), w, label="adversarial")
        ax.set_xticks(range(2)); ax.set_xticklabels(["NORMAL", "PNEUMONIA"])
        ax.set_title("diagnosis probabilities"); ax.legend(); ax.grid(axis="y", alpha=0.3)
        plt.show()

def _update(*_):
    render(img_picker.value, eps_picker.value)

img_picker.observe(_update, names="value")
eps_picker.observe(_update, names="value")

display(widgets.HBox([img_picker, eps_picker]))
display(out)
render(img_picker.value, eps_picker.value)

## 6. How badly does the attack break the detector? (all 624 test X-rays)

Same idea as the digits sweep: fix an epsilon, corrupt every test X-ray with FGSM, measure accuracy. `eps = 0.0` must reproduce the clean accuracy (integrity check).

In [ ]:
EPSILONS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4]

def accuracy_under_attack(loader, epsilon):
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE).view(-1)
        adv = images if epsilon == 0 else fgsm_attack(model, images, labels, epsilon)
        with torch.no_grad():
            correct += (model(adv).argmax(1) == labels).sum().item()
        total += labels.numel()
    return correct / total

results = [(eps, accuracy_under_attack(test_loader, eps)) for eps in tqdm(EPSILONS)]
for eps, acc in results:
    print(f"eps={eps:4.2f}  accuracy={acc * 100:5.2f}%")

assert abs(results[0][1] - CLEAN_ACC) < 0.005, "integrity check failed"

with open(ART / "metrics" / "accuracy_vs_epsilon.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["epsilon", "accuracy"]); w.writerows(results)

eps_x = [e for e, _ in results]; acc_y = [a * 100 for _, a in results]
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(eps_x, acc_y, marker="o", color="darkred")
for e, a in zip(eps_x, acc_y):
    ax.annotate(f"{a:.0f}%", (e, a), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
ax.set_xlabel("epsilon (invisible perturbation budget)")
ax.set_ylabel("diagnostic accuracy (%)")
ax.set_title("FGSM: pneumonia detector accuracy vs attack strength")
ax.set_ylim(0, 102); ax.grid(alpha=0.3)
fig.savefig(ART / "figures" / "accuracy_vs_epsilon.png", dpi=150)
plt.show()

## 7. Targeted attack — choose the wrong diagnosis on purpose

Instead of "make it wrong", demand a specific wrong answer. How often can we turn NORMAL into PNEUMONIA, and vice versa?

In [ ]:
def targeted_asr(source, target, eps):
    idxs = np.where(te_y == source)[0]
    loader = DataLoader(Subset(test_ds, idxs.tolist()), batch_size=128)
    hit = n = 0
    for images, labels in loader:
        images = images.to(DEVICE); labels = labels.to(DEVICE).view(-1)
        adv = fgsm_attack(model, images, torch.full_like(labels, target), eps, targeted=True)
        with torch.no_grad():
            hit += (model(adv).argmax(1) == target).sum().item()
        n += labels.numel()
    return hit / n, n

T_EPS = 0.3
r1, n1 = targeted_asr(0, 1, T_EPS)
r2, n2 = targeted_asr(1, 0, T_EPS)
print(f"targeted NORMAL -> PNEUMONIA: {r1 * 100:.1f}% of {n1} healthy X-rays forced into a false alarm (eps={T_EPS})")
print(f"targeted PNEUMONIA -> NORMAL: {r2 * 100:.1f}% of {n2} sick X-rays forced to 'healthy' (eps={T_EPS})")

with open(ART / "metrics" / "targeted_asr.json", "w") as f:
    json.dump({"epsilon": T_EPS,
               "normal_to_pneumonia_asr": r1, "n_normal": n1,
               "pneumonia_to_normal_asr": r2, "n_pneumonia": n2}, f, indent=2)

## 8. Bonus — PGD vs FGSM (same budget, stronger attack)

FGSM = one big step. PGD = 20 small steps, projected back into the budget after each. Medical note: the stronger attack needs no extra information — same model access, just more compute.

In [ ]:
def pgd_attack(model, images, labels, epsilon, alpha=None, steps=20):
    alpha = alpha if alpha is not None else epsilon / 4  # step scales with budget
    original = images.clone().detach()
    delta = torch.zeros_like(original).uniform_(-epsilon, epsilon)
    delta = ((original + delta).clamp(0, 1) - original).detach().requires_grad_(True)
    for _ in range(steps):
        loss = nn.functional.cross_entropy(model(original + delta), labels)
        (grad,) = torch.autograd.grad(loss, delta)
        with torch.no_grad():
            delta.add_(alpha * grad.sign())
            delta.clamp_(-epsilon, epsilon)
            delta.clamp_(min=-original, max=1 - original)
    return (original + delta).detach()

COMPARE_EPS = [0.0, 0.1, 0.2, 0.3]
fgsm_acc = [accuracy_under_attack(test_loader, e) for e in COMPARE_EPS]

pgd_acc = []
for e in tqdm(COMPARE_EPS, desc="PGD"):
    correct = total = 0
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE).view(-1)
        adv = images if e == 0 else pgd_attack(model, images, labels, e)
        with torch.no_grad():
            correct += (model(adv).argmax(1) == labels).sum().item()
        total += labels.numel()
    pgd_acc.append(correct / total)

for e, f, p in zip(COMPARE_EPS, fgsm_acc, pgd_acc):
    print(f"eps={e:4.2f}  FGSM {f * 100:5.2f}%  |  PGD(20 steps) {p * 100:5.2f}%")

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(COMPARE_EPS, [a * 100 for a in fgsm_acc], marker="o", label="FGSM (1 step)")
ax.plot(COMPARE_EPS, [a * 100 for a in pgd_acc], marker="s", label="PGD (20 steps)")
ax.set_xlabel("epsilon"); ax.set_ylabel("diagnostic accuracy (%)")
ax.set_title("Same budget, stronger attack")
ax.set_ylim(0, 102); ax.grid(alpha=0.3); ax.legend()
fig.savefig(ART / "figures" / "fgsm_vs_pgd.png", dpi=150)
plt.show()

## 9. Optional — test YOUR OWN chest X-ray

Run everything above first (so the model is trained), then:

1. Change `RUN_OWN_XRAY = False` to `True` below
2. Run **just this cell** (click it, press **Shift+Enter**)
3. Click the **"Choose Files"** button that appears and pick a chest X-ray image from your Mac

Works with a photo or a saved image. Tips: straight-on shot, dark background, whole chest visible, no glare, crop away the black frame. Don't have one handy? Save any chest X-ray from Google Images and try it.

**Expectation management:** this model learned from children's X-rays from one specific dataset. Your image will come from a different source, so the clean prediction may already be wrong — that's a "domain gap", not the attack. Still fun to watch the attack flip whatever the model says.

In [ ]:
RUN_OWN_XRAY = False  # <- set True, then run JUST this cell and pick a file

if RUN_OWN_XRAY:
    from google.colab import files
    from PIL import Image

    def to_model_format(img):
        img = img.convert("L")  # grayscale
        arr = np.array(img).astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255.0  # stretch contrast
        img = Image.fromarray(arr.astype(np.uint8))
        mask = np.array(img) > 25                      # crop away black borders
        ys, xs = np.where(mask)
        if len(xs) > 0:
            img = img.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))
        img = img.resize((28, 28), Image.LANCZOS)      # same size as training data
        return to_tensor(img)                          # (1, 28, 28) in [0, 1]

    uploaded = files.upload()
    if uploaded:
        name = next(iter(uploaded))
        x_up = to_model_format(Image.open(io.BytesIO(uploaded[name]))).unsqueeze(0).to(DEVICE)
        p_up = predict_probs(x_up)
        adv_up = fgsm_attack(model, x_up, p_up.argmax().reshape(1), 0.3)
        p_adv_up = predict_probs(adv_up)

        fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
        axes[0].imshow(x_up[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
        axes[0].set_title(f"your X-ray (as the model sees it)\nmodel: {CLASS_NAMES[p_up.argmax().item()]} ({p_up.max() * 100:.1f}%)", fontsize=9)
        axes[1].imshow(((adv_up - x_up)[0, 0] / 0.3).cpu(), cmap="gray", vmin=-1, vmax=1)
        axes[1].set_title("FGSM noise (amplified)", fontsize=9)
        axes[2].imshow(adv_up[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
        axes[2].set_title(f"adversarial\nmodel: {CLASS_NAMES[p_adv_up.argmax().item()]} ({p_adv_up.max() * 100:.1f}%)", fontsize=9)
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        print(f"clean diagnosis: {CLASS_NAMES[p_up.argmax().item()]}  ->  after attack: {CLASS_NAMES[p_adv_up.argmax().item()]}")
else:
    print("skipped — set RUN_OWN_XRAY = True and run this cell alone to try your own X-ray")

## 10. Example gallery + save your artifacts

A single image you can save/share: four sick X-rays the model correctly caught — each paired with its corrupted twin that the model now clears as NORMAL.

In [ ]:
N_GALLERY = 4
pairs = []
for i in pneumonias[:400]:
    img, _ = test_ds[int(i)]
    x = img.unsqueeze(0).to(DEVICE)
    if predict_probs(x).argmax().item() != 1:
        continue
    adv = fgsm_attack(model, x, torch.tensor([1], device=DEVICE), EPS_DEMO)
    if predict_probs(adv).argmax().item() == 0:
        pairs.append((x, adv))
    if len(pairs) == N_GALLERY:
        break

if pairs:
    fig, axes = plt.subplots(2, len(pairs), figsize=(3 * len(pairs), 6))
    for j, (x, adv) in enumerate(pairs):
        pc, pa = predict_probs(x), predict_probs(adv)
        axes[0, j].imshow(x[0, 0].cpu(), cmap="gray", vmin=0, vmax=1); axes[0, j].axis("off")
        axes[0, j].set_title(f"true PNEUMONIA\nmodel: PNEUMONIA ({pc.max() * 100:.0f}%)", fontsize=9)
        axes[1, j].imshow(adv[0, 0].cpu(), cmap="gray", vmin=0, vmax=1); axes[1, j].axis("off")
        axes[1, j].set_title(f"+ invisible noise (eps={EPS_DEMO})\nmodel: NORMAL ({pa.max() * 100:.0f}%)", fontsize=9)
    fig.suptitle("Top: real X-rays. Bottom: attacked twins the detector calls healthy.", fontsize=11)
    plt.tight_layout()
    fig.savefig(ART / "figures" / "attack_gallery.png", dpi=150)
    plt.show()
    print(f"saved -> {ART / 'figures' / 'attack_gallery.png'}")

print()
for p in sorted(ART.rglob("*")):
    if p.is_file():
        print(p)

if not USE_DRIVE:
    print("\nColab wipes /content when the session ends — set USE_DRIVE = True in setup, or uncomment:")
    # import shutil
    # from google.colab import files
    # shutil.make_archive("/content/artifacts", "zip", ART)
    # files.download("/content/artifacts.zip")

## Why this matters (and why it doesn't doom medical AI)

- Medical imaging AI is a real screening aid, and regulators (FDA/CE) increasingly expect **robustness testing** — exactly the sweeps you just ran — before deployment.
- The attack assumed **full access to the model's internals** (white-box). Real clinical deployments differ: images go through hospital pipelines (compression, resizing, different scanners), radiologists review "AI-suspicious" cases, and models are often ensembles — all of which make pure white-box noise harder in practice. But the robustness gap is real and measured, not hypothetical.
- You just produced the core evidence: a few percent of pixel change, invisible to a radiologist, can move diagnostic accuracy from ~90% to near coin-flip at eps=0.3.

**Next steps if you continue:**
1. **Adversarial training** — retrain with FGSM samples mixed in; see how much the curve holds up (the standard defense).
2. **Higher resolution** — modern X-ray models use 224x224+; MedMNIST supports `size=224` (re-run this notebook with 2 lines changed).
3. **Bigger clinical tasks** — CheXpert / NIH ChestX-ray14 (14 diseases per X-ray), Kaggle pneumonia dataset.
4. **Adversarial patches** — instead of spoofing every pixel, find one small sticker that flips any X-ray.

The digits version of this project lives in `FGSM_MNIST_Colab.ipynb` if you want to compare the two.